# Ablation 1 — No Augmentation
**Manufacturing Defect Detection | INTELSYS AY2526**

**What changed:** `RandomHorizontalFlip` and `RandomVerticalFlip` removed from `train_transform`. Everything else is identical to the baseline.

**What to observe:** If test F1 drops below baseline (0.9317), augmentation is contributing meaningfully to generalization.

In [ ]:
#CELL 1 — Mount and authenticate
!pip install kaggle -q
from google.colab import files
files.upload()

In [ ]:
#CELL 2 — Download dataset
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d yidazhang07/bridge-cracks-image -p /content/data --unzip

In [ ]:
#CELL 3 — Imports
import os, random, json
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
# Cell 4 — Custom dataset class
CLASS_MAP = {
    "MT_Blowhole": 0, "MT_Break": 1, "MT_Crack": 2,
    "MT_Fray": 3, "MT_Free": 4, "CF_Crack": 5
}
data_root_mt = "/content/data/Magnetic-Tile-Defect"
data_root_cf = "/content/data/CrackForest/image/image"
valid_extensions = (".jpg", ".jpeg", ".png", ".bmp")

class DefectDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, label

all_samples = []
for cls_name, label in CLASS_MAP.items():
    cls_path = data_root_cf if cls_name == "CF_Crack" else os.path.join(data_root_mt, cls_name, "Imgs")
    for f in os.listdir(cls_path):
        if f.lower().endswith(valid_extensions):
            all_samples.append((os.path.join(cls_path, f), label))
random.shuffle(all_samples)
print(f"Total samples: {len(all_samples)}")

In [ ]:
# Cell 5 — ABLATION 1: No Augmentation
# RandomHorizontalFlip + RandomVerticalFlip REMOVED from train_transform
# All other settings identical to baseline

total = len(all_samples)
train_end, val_end = int(0.70*total), int(0.85*total)
train_samples = all_samples[:train_end]
val_samples   = all_samples[train_end:val_end]
test_samples  = all_samples[val_end:]
print(f"Train: {len(train_samples)}, Val: {len(val_samples)}, Test: {len(test_samples)}")

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    # RandomHorizontalFlip() <-- REMOVED
    # RandomVerticalFlip()   <-- REMOVED
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
train_ds = DefectDataset(train_samples, transform=train_transform)
val_ds   = DefectDataset(val_samples,   transform=val_transform)
test_ds  = DefectDataset(test_samples,  transform=val_transform)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False)

In [ ]:
# Cell 6 — Load ResNet18 (Adam — same as baseline)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, len(CLASS_MAP))
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
print("Optimizer: Adam lr=1e-4 | Augmentation: DISABLED")

In [ ]:
# Cell 7 — Training loop
NUM_EPOCHS = 5
train_losses, val_losses, val_f1s = [], [], []

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    train_loss = running_loss / len(train_loader)
    train_losses.append(train_loss)

    model.eval()
    val_loss, all_preds, all_labels = 0.0, [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            val_loss += criterion(outputs, labels).item()
            all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    val_loss /= len(val_loader)
    val_losses.append(val_loss)
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    val_f1s.append(f1)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val F1: {f1:.4f}")

In [ ]:
# Cell 8 — Learning curves
plt.figure(figsize=(12, 4))
plt.subplot(1,2,1)
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.title("Loss Curves — Ablation 1: No Augmentation"); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()
plt.subplot(1,2,2)
plt.plot(val_f1s, label="Val Macro-F1", color="green")
plt.title("Val Macro-F1 — Ablation 1: No Augmentation"); plt.xlabel("Epoch"); plt.ylabel("F1"); plt.legend()
plt.tight_layout()
plt.savefig("FinAblation1_learning_curves.png", dpi=150)
plt.show()
print(f"Best Val F1: {max(val_f1s):.4f}")

In [ ]:
# Cell 9 — Test confusion matrix
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        all_preds.extend(model(imgs).argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
test_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
print(f"Test Macro-F1: {test_f1:.4f}")
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=list(CLASS_MAP.keys()))
fig, ax = plt.subplots(figsize=(8,8))
disp.plot(ax=ax, xticks_rotation=45)
plt.title("Confusion Matrix — Ablation 1: No Augmentation")
plt.tight_layout()
plt.savefig("FinAblation1_confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
# Cell 10 — Save metrics
metrics = {
    "ablation": "no_augmentation",
    "train_losses": train_losses,
    "val_losses": val_losses,
    "val_f1s": val_f1s,
    "test_macro_f1": test_f1,
    "seed": SEED, "epochs": NUM_EPOCHS, "batch_size": 32,
    "model": "ResNet18", "num_classes": len(CLASS_MAP)
}
with open("FinAblation1_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Saved — Test Macro-F1: {test_f1:.4f}")

In [ ]:
# Cell 11 — Download outputs
from google.colab import files
files.download("FinAblation1_learning_curves.png")
files.download("FinAblation1_confusion_matrix.png")
files.download("FinAblation1_metrics.json")